# 00 - Orquestación del Pipeline

**Objetivo:** Ejecutar todas las etapas del pipeline de forma automática

**Pasos:**
1. Extracción (01_Extraccion.ipynb)
2. Validación (02_Validacion.ipynb)
3. Cuarentena (03_Cuarentena.ipynb)

## Ejecución del Pipeline

Este notebook orquesta toda la ingesta de datos. Para ejecutar:

1. **Ejecuta cada notebook en orden:**
   - `01_Extraccion.ipynb` - Extrae datos y crea tabla RAW
   - `02_Validacion.ipynb` - Valida calidad de datos
   - `03_Cuarentena.ipynb` - Aísla registros con errores

2. **O ejecuta con nbclient (automático):**

In [ ]:
import subprocess
from pathlib import Path
from datetime import datetime

notebooks = [
    "01_Extraccion.ipynb",
    "02_Validacion.ipynb",
    "03_Cuarentena.ipynb"
]

current_dir = Path.cwd()

print("\n" + "="*70)
print("EJECUTANDO PIPELINE - INGESTA DE DATOS")
print("="*70)
print(f"Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

for idx, notebook in enumerate(notebooks, 1):
    notebook_path = current_dir / notebook
    
    print(f"PASO {idx}: {notebook}")
    print("-" * 70)
    
    try:
        # Ejecutar notebook con nbclient
        result = subprocess.run(
            ["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace", str(notebook_path)],
            capture_output=True,
            text=True,
            timeout=300
        )
        
        if result.returncode == 0:
            print(f"✓ {notebook} ejecutado correctamente\n")
        else:
            print(f"✗ Error al ejecutar {notebook}")
            print(f"  stderr: {result.stderr}\n")
    except Exception as e:
        print(f"✗ Excepción al ejecutar {notebook}: {str(e)}\n")

print("="*70)
print(f"Fin: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("✓ PIPELINE COMPLETADO")
print("="*70)

## Verificar Resultados

In [ ]:
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from conexion_sqlserver import obtener_engines

engine_origen, engine_destino = obtener_engines()

# Consultar tablas
query_raw = "SELECT COUNT(*) as cantidad FROM raw.invoice_lines"
query_validacion = "SELECT COUNT(*) as cantidad FROM raw.validacion_calidad"
query_cuarentena = "SELECT COUNT(*) as cantidad FROM quarantine.invoice_lines"

count_raw = pd.read_sql_query(query_raw, engine_destino).iloc[0]['cantidad']
count_validacion = pd.read_sql_query(query_validacion, engine_destino).iloc[0]['cantidad']
count_cuarentena = pd.read_sql_query(query_cuarentena, engine_destino).iloc[0]['cantidad']

print("\n" + "="*70)
print("RESULTADOS EN BASE DE DATOS")
print("="*70)
print(f"raw.invoice_lines:        {count_raw} registros")
print(f"raw.validacion_calidad:   {count_validacion} registros")
print(f"quarantine.invoice_lines: {count_cuarentena} registros")
print("="*70)

## Ver Archivos Generados

In [ ]:
from pathlib import Path

datos_dir = Path.cwd() / "datos"

print("\n" + "="*70)
print("ARCHIVOS GENERADOS")
print("="*70)

for subdir in ['bronze', 'cuarentena']:
    path = datos_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"\n{subdir}/ ({len(files)} archivos)")
        for f in files:
            size_kb = f.stat().st_size / 1024
            print(f"  - {f.name} ({size_kb:.1f} KB)")

print("\n" + "="*70)